# 05 — Risk-Based Pricing

## Purpose
Calculate indicative all-in lending rates by building up from the risk-free rate through funding costs, expected loss spreads, and risk-profile overlays.

**Bank context:** Loan pricing in commercial lending is not arbitrary — it is driven by a structured model that accounts for:
- Cost of funds (base rate + funding spread)
- Operating costs (servicing, admin)
- Capital costs (ROE hurdle driven by regulatory capital requirements)
- Expected loss (PD x LGD priced into the margin)
- Risk overlays (scorecard, leverage, liquidity, serviceability)

**Course alignment:**
- Commercial Ready Module 6: Interest Rate Components (Base Rate, Customer Margin, Treasury Margin, Line Fees)
- Excel model: `Risk_Based_Pricing` sheet

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_sme_dataset
from src.ratio_engine import calculate_ratios
from src.credit_scorecard import score_borrower, portfolio_scorecard_summary
from src.merton_pd import borrower_merton_analysis, portfolio_merton_summary
from src.working_capital import analyse_working_capital
from src.risk_based_pricing import compute_pricing, pricing_waterfall_table

pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

df = generate_sme_dataset(n_borrowers=80, seed=42)
df_r = calculate_ratios(df)

## Reference Borrower — Pricing Waterfall

Build up the all-in rate component by component.

In [ ]:
# Get all inputs for pricing
sc = score_borrower(df_r, borrower_id=0)
merton = borrower_merton_analysis(df, borrower_id=0)
fy0 = df_r[(df_r['borrower_id']==0) & (df_r['period']=='FY0')].iloc[0]
wc = analyse_working_capital(fy0)

pricing = compute_pricing(
    facility_amount=fy0['total_debt'],
    pd_value=merton['pd'],
    pvel=merton['pvel'],
    weighted_score=sc['weighted_score'],
    debt_to_ebitda=fy0['debt_to_ebitda'],
    wc_flag=wc['flag'],
    icr=fy0['icr'],
    dscr=fy0['dscr'],
)

waterfall = pricing_waterfall_table(pricing)

print('='*70)
print('RISK-BASED PRICING — Example AU SME Pty Ltd')
print(f'Facility: ${pricing["facility_amount"]:,.0f}')
print(f'Indicative All-in Rate: {pricing["all_in_rate"]:.2%}')
print('='*70)

# Format for display
wf = waterfall.copy()
wf['Rate'] = wf['Rate'].apply(lambda x: f'{x:.4%}' if isinstance(x, (int, float)) else x)
display(wf)

In [ ]:
# Pricing waterfall bar chart
components = [
    ('Base rate', pricing['base_rate']),
    ('Funding', pricing['funding_spread']),
    ('Operating', pricing['operating_spread']),
    ('Capital', pricing['capital_spread']),
    ('PVEL', pricing['pvel_spread']),
    ('Score', pricing['score_overlay']),
    ('PD', pricing['pd_overlay']),
    ('Leverage', pricing['leverage_overlay']),
    ('Liquidity', pricing['liquidity_overlay']),
    ('Coverage', pricing['coverage_overlay']),
]

labels = [c[0] for c in components]
values = [c[1] * 100 for c in components]  # Convert to percentage

fig, ax = plt.subplots(figsize=(14, 6))

# Stacked waterfall
cumulative = 0
colors = ['#1565C0', '#1976D2', '#2196F3', '#42A5F5',
          '#FF5722', '#FF9800', '#FFC107', '#4CAF50', '#8BC34A', '#CDDC39']

for i, (label, val) in enumerate(zip(labels, values)):
    ax.bar(i, val, bottom=cumulative, color=colors[i], alpha=0.85, edgecolor='white', linewidth=0.5)
    if val > 0.05:  # Only label visible bars
        ax.text(i, cumulative + val/2, f'{val:.2f}%', ha='center', va='center', fontsize=9, fontweight='bold')
    cumulative += val

# All-in rate line
ax.axhline(pricing['all_in_rate'] * 100, color='red', linestyle='--', linewidth=2,
           label=f'All-in rate: {pricing["all_in_rate"]:.2%}')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Rate (%)')
ax.set_title('Risk-Based Pricing Build-Up', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/figures/05_pricing_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## Portfolio Pricing Distribution

How do indicative rates vary across the portfolio by credit grade?

In [ ]:
# Price all borrowers
port_pd = portfolio_merton_summary(df)
port_sc = portfolio_scorecard_summary(df_r)

pricing_results = []
for bid in df_r['borrower_id'].unique():
    try:
        sc_b = score_borrower(df_r, bid)
        if 'error' in sc_b:
            continue
        m_b = borrower_merton_analysis(df, bid)
        fy0_b = df_r[(df_r['borrower_id']==bid) & (df_r['period']=='FY0')].iloc[0]
        wc_b = analyse_working_capital(fy0_b)
        
        p = compute_pricing(
            facility_amount=fy0_b['total_debt'],
            pd_value=m_b['pd'],
            pvel=m_b['pvel'],
            weighted_score=sc_b['weighted_score'],
            debt_to_ebitda=fy0_b['debt_to_ebitda'],
            wc_flag=wc_b['flag'],
            icr=fy0_b['icr'],
            dscr=fy0_b['dscr'],
        )
        pricing_results.append({
            'borrower_id': bid,
            'grade': sc_b['grade'],
            'all_in_rate': p['all_in_rate'],
            'total_margin': p['total_margin'],
            'pd': m_b['pd'],
        })
    except Exception:
        continue

pricing_df = pd.DataFrame(pricing_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rate by grade
grade_order = ['A', 'B', 'C', 'D']
grade_colors = {'A': '#4CAF50', 'B': '#8BC34A', 'C': '#FF9800', 'D': '#F44336'}

for grade in grade_order:
    gdf = pricing_df[pricing_df['grade'] == grade]
    if len(gdf) > 0:
        axes[0].scatter([grade] * len(gdf), gdf['all_in_rate'] * 100,
                       color=grade_colors[grade], alpha=0.6, s=50)
        axes[0].scatter([grade], [gdf['all_in_rate'].median() * 100],
                       color=grade_colors[grade], s=200, marker='D', edgecolors='black', zorder=5)

axes[0].set_title('Indicative Rate by Internal Grade', fontweight='bold')
axes[0].set_xlabel('Internal Grade')
axes[0].set_ylabel('All-in Rate (%)')

# Rate vs PD scatter
axes[1].scatter(pricing_df['pd'] * 100, pricing_df['all_in_rate'] * 100,
               c=pricing_df['grade'].map(grade_colors), alpha=0.6, s=50)
axes[1].set_title('All-in Rate vs Merton PD', fontweight='bold')
axes[1].set_xlabel('Merton PD (%)')
axes[1].set_ylabel('All-in Rate (%)')

plt.tight_layout()
plt.savefig('../outputs/figures/05_portfolio_pricing.png', dpi=150, bbox_inches='tight')
plt.show()

print('Median all-in rate by grade:')
print(pricing_df.groupby('grade')['all_in_rate'].median().apply(lambda x: f'{x:.2%}').to_string())

---

## Key Takeaways

1. **Risk-based pricing is not a single number** — it is built up from base rate + funding + capital + expected loss + risk overlays
2. **Better credit quality = lower rate** — Grade A borrowers receive the tightest pricing
3. **PVEL spread is the core structural component** — it directly prices the expected loss into the margin
4. **Overlays act as policy buffers** — they capture risks not fully reflected in the PD model
5. **This is how banks calculate ROE** — the margin above cost of funds must cover expected losses AND deliver a target return on regulatory capital

**Next:** [06_Full_Credit_Paper_Walkthrough.ipynb](06_Full_Credit_Paper_Walkthrough.ipynb)